# Examine Analysis: 

Notebook that used the classes from attpc_analysis for the analysis notebooks

In [ ]:
from spyral_utils.nuclear import NuclearDataMap
from spyral_utils.nuclear.target import SolidTarget, GasTarget, load_target
from spyral_utils.constants import AMU_2_MEV, QBRHO_2_P
from pathlib import Path
import numpy as np
import polars as pl



from spyral_utils.plot.histogram import Histogrammer
import matplotlib.pyplot as plt 
import matplotlib as mpl
cmap = mpl.colormaps["rainbow"]

%matplotlib widget
plt.close("all")
# Import the ion chamber properties 
nuclear_map = NuclearDataMap()


# This is for the meantime -------------------------------------------------------
# The final motivation is to make it pip installeable 
import sys
src_path = Path.cwd().parents[1]
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --------------------------------------------------------------------------------
from attpc_analysis.src.load.io import RunSolverDataLoader, RunEstimationDataLoader
from attpc_analysis.src.data_models.physics import DataReactionKinematics
from attpc_analysis.src.detector_settings.energy_loss_setup import EnergyLossSetup

# Import the ion chamber properties 
nuclear_map = NuclearDataMap()


In [ ]:
import vector 
def vector_direction(v1: vector, v2: vector):
    #Get the direction of the unitary vector with respect two vectors
    v1_unit = vector.array({"px":v1.px/v1.p, "py": v1.py/v1.p, "pz":v1.pz/v1.p})
    v2_unit = vector.array({"px":v2.px/v2.p, "py": v2.py/v2.p, "pz":v2.pz/v2.p})
    return v1_unit.dot(v2_unit)

def angular_distribution(parent_vector: vector, original_vector: vector, observed_vector: vector, interest: NuclearDataMap): #targetlike_vector: vector , parent_vector: vector, target_vector: vector):
    """
    Parent vector: Vector for boosting
    Original vector: Target or Beam vector (parent particle before the reaction) 
    Observed vector: Targetlike or Beamlike vector (daughter particle after the reaction) 

    [Create]
    interest vector: Nucleus of interest (for excitation energy)

    """
    interest_vector= parent_vector - observed_vector 

    #Convert to the CM frame
    original_vector_cm = original_vector.boostCM_of(parent_vector)
    observed_vector_cm = observed_vector.boostCM_of(parent_vector)
    interest_vector_cm = interest_vector.boostCM_of(parent_vector)

    #Get angle in the real CM. 
    cos_polar_cm = vector_direction(original_vector_cm, observed_vector_cm)
    excitation_energy = interest_vector.mass - interest.mass
    return (cos_polar_cm, excitation_energy)

Load event from jason file

In [ ]:
from json import load
exp_tag = "a2066"
nuclear_map = NuclearDataMap()
path_to_config = Path(f"../../notebooks/input_parameters/{exp_tag}.json")
with open(path_to_config,"r") as json_file:
    json_data = load(json_file)
    # Get keys from json and use them as their variables 
    workspace_path = Path(json_data["workspace_path"])
    trace_path = Path(json_data["trace_path"])
    target_material_path = Path(json_data["target_path"])
    run_min = json_data["run_min"]
    run_max = json_data["run_max"]
    runs_to_skip = json_data["runs_to_skip"]
    ejectile = nuclear_map.get_data(json_data["ejectile"][0], json_data["ejectile"][1])
    projectile = nuclear_map.get_data(json_data["projectile"][0], json_data["projectile"][1])
    target = nuclear_map.get_data(json_data["target"][0], json_data["target"][1])
    ic_pressure = json_data["ic_pressure"]
    proj_energy_accel = json_data["beam_energy"]
    ic_min_val = json_data["ic_min_val"]
    ic_max_val = json_data["ic_max_val"]
solver_result_path = workspace_path / "InterpSolver"

# We calculate the residual for you
residual_z = target.Z + projectile.Z - ejectile.Z
residual_a = target.A + projectile.A - ejectile.A

if residual_z < 0:
    raise Exception(f"Illegal nuclei! Residual Z: {residual_z}")
if residual_a < 1:
    raise Exception(f"Illegal nuclei! Residual A: {residual_a}")

residual = nuclear_map.get_data(residual_z, residual_a)

In [ ]:
workspace_path

In [ ]:
# Get the beam energy for the active area of the detector
energy_loss_setup = EnergyLossSetup(
    beam_energy = proj_energy_accel,
    projectile = projectile,
    attpc_target_path_json= target_material_path
)

target_material = energy_loss_setup.active_gas_target_material()
proj_energy_start = energy_loss_setup.beam_energy_entrance_attpc()
proj_energy_stop = energy_loss_setup.beam_energy_exit_attpc()

print(f"Reaction: {target}({projectile}, {ejectile}){residual}")
print(f"Target material: {target_material.ugly_string}")
print(f"Accelerator Beam energy: {proj_energy_accel} MeV")
print(f"Beam energy range in AT-TPC: {proj_energy_start}-{proj_energy_stop} MeV")

In [ ]:
# This is for the meantime -------------------------------------------------------
# The final motivation is to make it pip installeable 
import sys
src_path = Path.cwd().parents[1]
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --------------------------------------------------------------------------------
from attpc_analysis.src.load.io import RunSolverDataLoader, RunEstimationDataLoader
from attpc_analysis.src.data_models.physics import DataReactionKinematics




# Date storage
results = {}
results["run"] = np.empty(0,int)
results["event"] = np.empty(0,int)
results["ex"] = np.empty(0,float)
results["polar_cm"] = np.empty(0,float)



# Add these for kinematics.extract_kinematics_values(results)
results["brho"] = np.empty(0,float)
results["kinetic_energy"] = np.empty(0,float)
results["polar"] = np.empty(0,float)
results["az"] = np.empty(0,float)

grammer = Histogrammer()
grammer.add_hist1d("ex", 200,(-2,30))


# Load class with general details of the runs
empty_files = []
solver_data = RunSolverDataLoader(
                solver_result_path=solver_result_path, 
                particle=ejectile,
                empty_files = empty_files
                )

for run in range(run_min, run_max+1):
    df = solver_data.load_solver(run)
    if df is None: continue 
    #------------------------------------------------------------------------------
    # Insert specific filters into the dataframe 
    # (Here)
    #------------------------------------------------------------------------------
    # brho = df.select("brho").to_numpy().flatten()
    # momentum = brho* float(ejectile.Z) * QBRHO_2_P
    # kinetic_energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0) - ejectile.mass
    # kientic_energy = kinetic_energy*slope + intercept 

    # momentum = np.sqrt((kinetic_energy + ejectile.mass)**2.0 - ejectile.mass**2.)
    # brho  = momentum / QBRHO_2_P /float(ejectile.Z)
    # df.replace_column(df.find_idx_by_name("brho"),brho)
    # print(df.columns)
    # break 
    kinematics = DataReactionKinematics(
        data = df, 
        projectile= projectile, 
        target = target, 
        targetlike = ejectile, 
        beamlike = residual, 
        beam_energy_entrace_tpc= proj_energy_start, 
    )
    # Clear data frame
    results["event"] = np.append(results["event"], df.select("event").to_numpy().flatten())
    df = None 
    
    #------------------------------------------------------------------------------
    # Do other things: example missing mass 
    # Insert specific actions that will be saved in arrays
    beamlike_vector = kinematics.missing_mass_technique(target_material)
    ex = beamlike_vector.mass - residual.mass
    # IMPROVE THIS ----------------------------------------------------------------------------!!!! 
    targetlike_vector = kinematics.construct_targetlike_vector()
    parent_vector = kinematics.construct_parent_vector(target_material)
    target_vector = kinematics.construct_target_vector()

    cos_polar_cm, ex_beamlike = angular_distribution(parent_vector, target_vector, targetlike_vector, residual)
    #------------------------------------------------------------------------------

    # Save information 
    results["run"] = np.append(results["run"],np.full(len(ex),run))
    
    results["ex"] = np.append(results["ex"],ex)
    results["polar_cm"] = np.append(results["polar_cm"],np.arccos(cos_polar_cm))

    kinematics.extract_kinematics_values(results)


    # fill histograms 
    grammer.fill_hist1d("ex",ex)
    

print("Files without data", empty_files)


In [ ]:
import vector 
# Add states here!
residual_excitations = np.array([0,0.4291])#,4.57,6.73,7.21,9.27,9.9,11.01,17]) #7Be
# residual_excitations = np.array([0,0.4291,4.57,6.73]) #6Be

ejectile = nuclear_map.get_data(1,2)
residual = nuclear_map.get_data(4,7)


# residual_excitations = np.array([0])

# Do the calculation in the center of mass
cm_angle_range = np.linspace(0, np.pi, 1000)
# Calculate envelope from range of beam energies
eject_kinematics = np.zeros((len(residual_excitations), len(cm_angle_range) * 2, 2))
# proj_energy_start = 90

# Upper limit (maximum beam energy)
beam_vec_max = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [np.sqrt(proj_energy_start * (proj_energy_start + 2.0 * projectile.mass))],
    "E": [proj_energy_start + projectile.mass]
})

parent_max = target_vector + beam_vec_max
parent_cm_max = parent_max.boostCM_of(parent_max)
for idx, ex in enumerate(residual_excitations):
    eject_e_cm_max = (ejectile.mass**2.0 - (residual.mass + ex)**2.0 + parent_cm_max.E**2.0) / (2.0 * parent_cm_max.E)
    eject_p_cm_max = np.sqrt(eject_e_cm_max**2.0 - ejectile.mass**2.0)
    eject_vec_cms_max = vector.array({
        "px": eject_p_cm_max * np.sin(np.pi- cm_angle_range),
        "py": np.zeros(len(cm_angle_range)),
        "pz": eject_p_cm_max * np.cos(np.pi- cm_angle_range),
        "E": np.full(len(cm_angle_range), eject_e_cm_max)
    })
    eject_vec_lab_max = eject_vec_cms_max.boost(parent_max)
    eject_kinematics[idx, :len(cm_angle_range), 0] = eject_vec_lab_max.theta
    eject_kinematics[idx, :len(cm_angle_range), 1] = eject_vec_lab_max.E - ejectile.mass
    # break


# Lower limit (minimum beam energy)
beam_vec_min = vector.array({
    "px": [0.0],
    "py": [0.0],
    "pz": [np.sqrt(proj_energy_stop * (proj_energy_stop + 2.0 * projectile.mass))],
    "E": [proj_energy_stop + projectile.mass]
})
parent_min = target_vector + beam_vec_min
parent_cm_min = parent_min.boostCM_of(parent_min)
for idx, ex in enumerate(residual_excitations):
    eject_e_cm_min = (ejectile.mass**2.0 - (residual.mass + ex)**2.0 + parent_cm_min.E**2.0) / (2.0 * parent_cm_min.E)
    eject_p_cm_min = np.sqrt(eject_e_cm_min**2.0 - ejectile.mass**2.0)
    eject_vec_cms_min = vector.array({
        "px": eject_p_cm_min * np.sin(np.pi-cm_angle_range),
        "py": np.zeros(len(cm_angle_range)),
        "pz": eject_p_cm_min * np.cos(np.pi-cm_angle_range),
        "E": np.full(len(cm_angle_range), eject_e_cm_min)
    })
    eject_vec_lab_min = eject_vec_cms_min.boost(parent_min)
    # Note that lower limit is "flipped". We plot the band as a polygon, so need to continue from large lab angles to small lab angles
    eject_kinematics[idx, len(cm_angle_range):, 0] = np.flip(eject_vec_lab_min.theta)
    eject_kinematics[idx, len(cm_angle_range):, 1] = np.flip(eject_vec_lab_min.E - ejectile.mass)

In [ ]:
# open deuteron file 
# deuterons = pl.read_csv("./../../personal_analysis/tail_elastic_dent.csv") # events with a+3He+d
deuterons = pl.read_csv("./high_energy_deuterons.csv")

red_points = {}
red_points["polar"] = np.empty(0,float)
red_points["brho"] = np.empty(0,float)
red_points["polar_cm"] =  np.empty(0,float)
red_points["ex"] =  np.empty(0,float)
red_points["kinetic"] =  np.empty(0,float)
for run in np.unique(deuterons["run"]): 
    mask = deuterons["run"] == run 
    events = deuterons.select("event").to_numpy()[mask]
    for event in np.unique(events):
        mask = (results["run"] == run) & (results["event"]==event)
        red_points["polar"] = np.append(red_points["polar"], results["polar"][mask])
        red_points["brho"] = np.append(red_points["brho"], results["brho"][mask])
        red_points["polar_cm"]= np.append(red_points["polar_cm"], results["polar_cm"][mask])
        red_points["ex"]= np.append(red_points["ex"], results["ex"][mask])
        red_points["kinetic"]= np.append(red_points["kinetic"], results["kinetic_energy"][mask])

In [ ]:
len(red_points["polar"]) # 1785 originally, new prints 732. 

In [ ]:
# Make angle cuts 
make_cuts = False
show_kinematic_line = True 

angle = 60
step = 10
# constrain kinetic energy 
# mask_ke = (np.rad2deg(np.pi-cm_angle_range)> angle) & (np.rad2deg(np.pi-cm_angle_range)<(angle+step))
# average_ke = np.mean(np.pi-cm_angle_range)

mask = (results["polar"]> np.deg2rad(40) ) & (results["polar"]< np.deg2rad(50)) & (results["kinetic_energy"]< 50  )  & (results["kinetic_energy"]> 26  )#gate

# mask = (results["polar"]>np.deg2rad(angle)) & (results["polar"]<np.deg2rad(angle+step)) # lab cuts


# Real mask
# mask_ke = (np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0])> angle) & (np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0])<(angle+step))
# average_ke = np.mean(eject_kinematics[idx, :len(cm_angle_range), 1][mask_ke])
# mask = ( 
#     (results["polar"]>np.deg2rad(angle)) & (results["polar"]<np.deg2rad(angle+step)) # lab cuts 
#     & (results["kinetic_energy"]> average_ke-8) & (results["kinetic_energy"]< average_ke+8)
# )

if not make_cuts: 
    mask = np.full(len(results["ex"]), True)


plt.close("all")
fig, axs = plt.subplots(1,3,figsize=(16,6))
fig.suptitle(f"{nuclear_map.get_data(4,7).get_latex_rep()}({nuclear_map.get_data(1,2).get_latex_rep()},{nuclear_map.get_data(1,2).get_latex_rep()}){nuclear_map.get_data(4,7).get_latex_rep()}")

# Figure 1: Kinematics
ax = axs[0]
grammer.add_hist2d("kinematics", (200,200), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics", np.rad2deg(results["polar"][mask]), results["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
# ax.scatter(np.rad2deg(red_points["polar"]), red_points["kinetic"],s=3, color='red')
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")

if show_kinematic_line:
    cmap = mpl.colormaps["rainbow"]
    colors = cmap(np.linspace(0,1,len(residual_excitations)))

    for idx, ex in enumerate(residual_excitations):
        ax.plot(np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0]), eject_kinematics[idx, :len(cm_angle_range), 1], label=fr"$E_x$={ex:02} MeV",color=colors[idx])
        ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],color=colors[idx],alpha=0.2)#, label=f"Ex={ex:02} Band", alpha=0.2)

ax.grid()
ax.legend()
# Figure 2: Ex vs Polar (cm) 
ax = axs[1]
grammer.add_hist2d("ex_polarcm",(200,200), ((0,180),(-5,20)))
grammer.fill_hist2d("ex_polarcm",np.rad2deg(results["polar_cm"][mask]), results["ex"][mask])
hist = grammer.get_hist2d("ex_polarcm")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_ylim([-5,20])
ax.set_xlim([0,180])
# fig.colorbar(mesh)
ax.set_xlabel(r"$\theta_\text{cm}$ (deg)")
ax.set_ylabel(r"$E_x$ (MeV)")
ax.grid()

if show_kinematic_line: 
    for idx, ex in enumerate(residual_excitations):
        if ex>20: continue
        ax.axhline(ex,label=fr"$E_x$={ex:02} MeV",color=colors[idx] )


# Figure 3: Ex 
ax = axs[2]
min_bin, max_bin = -5 , 14
bin_size = 38
granularity = (max_bin - min_bin) / bin_size
# missing mass 
grammer.add_hist1d("ex", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex", results["ex"][mask])
hist  = grammer.get_hist1d("ex")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black',label='missing')
# ax.scatter(hist_x, hist.counts,s=10,color='black', label='missing')
# ax.stairs(hist.counts,edges=hist.bins,color='black')

ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"$E_x$ (MeV)")

if show_kinematic_line: 
    colors = cmap(np.linspace(0,1,len(residual_excitations)))
    for idx, ex in enumerate(residual_excitations):
        ax.axvline(ex,label=fr"$E_x$={ex:02} MeV",color=colors[idx] )

ax.legend()
ax.set_ylim([0,25000])
# ax.set_ylim([0,2000])
# ax.set_xlim([-5,4])
fig.tight_layout()
fig.savefig("delete.png")

In [ ]:
frame = {"run": results["run"][mask], "event": results["event"][mask]}
df = pl.DataFrame(frame)
# df.write_csv("./high_energy_deuterons.csv")

In [ ]:
# Calibration 
import lmfit


residual_excitations = np.array([0.])
step = 5
# angle_calibration = np.arange(45,70,step)
angle_calibration = np.arange(40,85,step)
centroid = np.empty(0,float)
kinetic_energy = np.empty(0,float)
plt.close("all")
for angle in angle_calibration:
    # constrain kinetic energy 
    mask_ke = (np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0])> angle) & (np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0])<(angle+step))
    average_ke = np.mean(eject_kinematics[idx, :len(cm_angle_range), 1][mask_ke])
    # Real mask
    mask = ( 
        (results["polar"]>np.deg2rad(angle)) & (results["polar"]<np.deg2rad(angle+step)) # lab cuts 
        & (results["kinetic_energy"]> average_ke-8) & (results["kinetic_energy"]< average_ke+8)
    )


    # Step 0: make sure the histogram is for the region of interest
    # plt.close("all")
    fig, ax = plt.subplots(1,1)
    min_bin, max_bin = -5 , 14
    bin_size = 38
    granularity = (max_bin - min_bin) / bin_size
    # missing mass 
    grammer.add_hist1d("ex", bin_size, (min_bin,max_bin))
    grammer.fill_hist1d("ex", results["ex"][mask])
    hist  = grammer.get_hist1d("ex")
    uncertainty = np.sqrt(np.maximum(hist.counts,1))
    hist_x = hist.bins[:-1] + 0.5*hist.bin_width
    ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black',label='missing')
    # Step 1: Make gausian fits in the lab angular distribution 
    # Add fit to the GS. 
    peak_gs = lmfit.models.GaussianModel(prefix="peak1_")
    peak_gs_bins, peak_gs_counts = hist.get_subrange((-5, 3))
    peak_gs_bins +=0.5*hist.bin_width
    peak_gs_weights = 1.0/np.sqrt(peak_gs_counts)
    peak_gs_weights[peak_gs_counts==0] = 1.0
    pars = peak_gs.guess(x=peak_gs_bins, data=peak_gs_counts, weights=peak_gs_weights)



    total_spectrum = peak_gs
    total_bins = hist.bins[:-1] + 0.5*hist.bin_width
    total_counts= hist.counts 
    total_weights = 1/np.sqrt(np.maximum(total_bins,1))
    total_spectrum = total_spectrum.fit(params=pars, x=total_bins, data= total_counts, weights = total_weights)
    comps = total_spectrum.eval_components(x=total_bins)
    print(total_spectrum.fit_report())

    # Create uniform fit 
    total_bins_fit = np.linspace(min_bin, max_bin,1000)
    comps_fit = total_spectrum.eval_components(x=total_bins_fit)
    total_result_fit = total_spectrum.eval(x=total_bins_fit)


    name_str = f"peak1_"
    value_string = r"$\bar{x}$: " + f"{total_spectrum.params[name_str+"center"].value:0.2f} MeV, " + r"$\sigma$ : " + f" {total_spectrum.params[name_str+"sigma"].value*1000:0.2f} keV"
    # ax.plot(total_bins_fit, comps_fit[name_str], label=f"x : {total_spectrum.params[name_str+"center"].value:0.2f} MeV, \\\\sigma : {total_spectrum.params[name_str+"sigma"].value:0.2f}", color=colors[n-1])
    ax.plot(total_bins_fit, comps_fit[name_str], label=value_string, color="purple")
    ax.plot(total_bins_fit, total_result_fit,color='black', label="Total Fit",linestyle='--')


    ax.legend()
    ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
    ax.set_xlabel(f"$E_x$ (MeV)")
    ax.set_title(f"{angle}"+r"$\leq \theta_{lab} \leq$"+ f"{angle+step}")

    # ax.set_xlim([-5,4])

    # append centroid info 
    centroid = np.append(centroid,total_spectrum.params[name_str+"center"].value )
    


In [ ]:
plt.close("all")
fig, ax = plt.subplots(1,1)

ax.scatter(angle_calibration+step/2, centroid) # centroid: exp data, kinetic_energy: real value
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel(r"$E_0$ Centroid Gauss. (MeV)")
ax.grid()

slope, intercept = np.polyfit(angle_calibration+step/2, centroid, 1)
print(slope,intercept)
ax.axline((0,intercept),slope=slope,color='red',linestyle='--')
ax.set_box_aspect(1)
ax.set_xlim([30,90])
ax.set_ylim([-3,1])

### Fitting the KE vs polar lab

In [ ]:
# Calibration 
import lmfit


residual_excitations = np.array([0.])
step = 2
# angle_calibration = np.arange(45,70,step)
angle_calibration = np.arange(40,90,step)
centroid = np.empty(0,float)
kinetic_energy = np.empty(0,float)

for angle in angle_calibration:
    mask = ( 
        (results["polar"]>np.deg2rad(angle)) & (results["polar"]<np.deg2rad(angle+step)) # lab cuts
        & (results["ex"]>-5) & (results["ex"]<2)#(results["ex"]<0.32)
    )

    mask_ke = (np.rad2deg(eject_kinematics[0, :len(cm_angle_range), 0])> angle) & (np.rad2deg(eject_kinematics[0, :len(cm_angle_range), 0])<(angle+step))
    print(sum(mask_ke))
    average_ke = np.mean(eject_kinematics[0, :len(cm_angle_range), 1][mask_ke])

    # Step 0: make sure the histogram is for the region of interest
    # plt.close("all")
    fig, ax = plt.subplots(1,1)
    min_bin, max_bin = np.max(average_ke-10,0), average_ke+10
    bin_size = 38
    granularity = (max_bin - min_bin) / bin_size
    print(min_bin,max_bin)
    # missing mass 
    grammer.add_hist1d("kinematics1d", bin_size, (min_bin,max_bin))
    grammer.fill_hist1d("kinematics1d", results["kinetic_energy"][mask])
    hist  = grammer.get_hist1d("kinematics1d")
    uncertainty = np.sqrt(np.maximum(hist.counts,1))
    hist_x = hist.bins[:-1] + 0.5*hist.bin_width
    ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black',label='missing')
    ax.axvline(average_ke,color='purple', linestyle='--')
    # Step 1: Make gausian fits in the lab angular distribution 
    # Add fit to the GS. 
    peak_gs = lmfit.models.GaussianModel(prefix="peak1_")
    peak_gs_bins, peak_gs_counts = hist.get_subrange((min_bin, max_bin))
    peak_gs_bins +=0.5*hist.bin_width
    peak_gs_weights = 1.0/np.sqrt(peak_gs_counts)
    peak_gs_weights[peak_gs_counts==0] = 1.0
    pars = peak_gs.guess(x=peak_gs_bins, data=peak_gs_counts, weights=peak_gs_weights)



    total_spectrum = peak_gs
    total_bins = hist.bins[:-1] + 0.5*hist.bin_width
    total_counts= hist.counts 
    total_weights = 1/np.sqrt(np.maximum(total_bins,1))
    total_spectrum = total_spectrum.fit(params=pars, x=total_bins, data= total_counts, weights = total_weights)
    comps = total_spectrum.eval_components(x=total_bins)
    print(total_spectrum.fit_report())

    # Create uniform fit 
    total_bins_fit = np.linspace(min_bin, max_bin,1000)
    comps_fit = total_spectrum.eval_components(x=total_bins_fit)
    total_result_fit = total_spectrum.eval(x=total_bins_fit)


    name_str = f"peak1_"
    value_string = r"$\bar{x}$: " + f"{total_spectrum.params[name_str+"center"].value:0.2f} MeV, " + r"$\sigma$ : " + f" {total_spectrum.params[name_str+"sigma"].value*1000:0.2f} keV"
    # ax.plot(total_bins_fit, comps_fit[name_str], label=f"x : {total_spectrum.params[name_str+"center"].value:0.2f} MeV, \\\\sigma : {total_spectrum.params[name_str+"sigma"].value:0.2f}", color=colors[n-1])
    ax.plot(total_bins_fit, comps_fit[name_str], label=value_string, color="purple")
    ax.plot(total_bins_fit, total_result_fit,color='black', label="Total Fit",linestyle='--')


    ax.legend()
    ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
    ax.set_xlabel(f"$E_x$ (MeV)")
    ax.set_title(f"{angle}"+r"$\leq \theta_{lab} \leq$"+ f"{angle+step}")
    # ax.set_xlim([-5,4])

    # append centroid info 
    centroid = np.append(centroid,total_spectrum.params[name_str+"center"].value )
    kinetic_energy = np.append(kinetic_energy, average_ke)
    # break

plt.close("all")

In [ ]:
plt.close("all")
fig, ax = plt.subplots(1,1)

ax.scatter(centroid, kinetic_energy) # centroid: exp data, kinetic_energy: real value
mask = angle_calibration == 70
# print(centroid[mask])
ax.scatter(centroid[mask],kinetic_energy[mask])
ax.set_xlabel(r"$KE$ Exp (MeV)")
ax.set_ylabel(r"$KE$ Theo (MeV)")
ax.grid()

slope, intercept = np.polyfit(centroid, kinetic_energy, 1)
print(slope,intercept)
ax.axline((0,intercept),slope=slope,color='red',linestyle='--')
ax.set_box_aspect(1)
# ax.set_xlim([0.3,1.3])
# ax.set_ylim([-0.2,50])

In [ ]:
print("slope", slope, "inter", intercept)

In [ ]:
plt.close("all")
fig, ax = plt.subplots(1,1)
fig.suptitle(f"{nuclear_map.get_data(4,7).get_latex_rep()}({nuclear_map.get_data(1,2).get_latex_rep()},{nuclear_map.get_data(1,2).get_latex_rep()}){nuclear_map.get_data(4,7).get_latex_rep()}")

# Figure 1: Kinematics
mask = np.full(len(results["ex"]), True)
grammer.add_hist2d("kinematics", (200,200), ((0,90),(0,70)))
dKE = results["kinetic_energy"][mask]*slope + intercept
grammer.fill_hist2d("kinematics", np.rad2deg(results["polar"][mask]), dKE)
hist = grammer.get_hist2d("kinematics")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")

if show_kinematic_line:
    cmap = mpl.colormaps["rainbow"]
    colors = cmap(np.linspace(0,1,len(residual_excitations)))

    for idx, ex in enumerate(residual_excitations):
        ax.plot(np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0]), eject_kinematics[idx, :len(cm_angle_range), 1], label=fr"$E_x$={ex:02} MeV",color=colors[idx])
        ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],color=colors[idx],alpha=0.2)#, label=f"Ex={ex:02} Band", alpha=0.2)

In [ ]:


# Date storage
results = {}
results["run"] = np.empty(0,int)
results["event"] = np.empty(0,int)
results["ex"] = np.empty(0,float)
results["polar_cm"] = np.empty(0,float)



# Add these for kinematics.extract_kinematics_values(results)
results["brho"] = np.empty(0,float)
results["kinetic_energy"] = np.empty(0,float)
results["polar"] = np.empty(0,float)
results["az"] = np.empty(0,float)

grammer = Histogrammer()
grammer.add_hist1d("ex", 200,(-2,30))


# Load class with general details of the runs
empty_files = []
solver_data = RunSolverDataLoader(
                solver_result_path=solver_result_path, 
                particle=ejectile,
                empty_files = empty_files
                )

for run in range(run_min, run_max+1):
    df = solver_data.load_solver(run)
    if df is None: continue 
    #------------------------------------------------------------------------------
    # Insert specific filters into the dataframe 
    # (Here)
    #------------------------------------------------------------------------------
    brho = df.select("brho").to_numpy().flatten()
    polar = df.select("polar").to_numpy().flatten()
    momentum = brho* float(ejectile.Z) * QBRHO_2_P
    kinetic_energy = np.sqrt(momentum**2.0 + ejectile.mass**2.0) - ejectile.mass
    # kethe= 13
    # polar=polar-(2*np.pi/4000)*(kinetic_energy-kethe)
    # df.replace_column(df.get_column_index("polar"),pl.Series("polar",polar))
    kinetic_energy = kinetic_energy*slope + intercept 
    momentum = np.sqrt((kinetic_energy + ejectile.mass)**2.0 - ejectile.mass**2.)
    brho  = momentum / QBRHO_2_P /float(ejectile.Z)
    df.replace_column(df.get_column_index("brho"),pl.Series("brho",brho))


    kinematics = DataReactionKinematics(
        data = df, 
        projectile= projectile, 
        target = target, 
        targetlike = ejectile, 
        beamlike = residual, 
        beam_energy_entrace_tpc= proj_energy_start, 
    )
    # Clear data frame
    df = None 
    
    #------------------------------------------------------------------------------
    # Do other things: example missing mass 
    # Insert specific actions that will be saved in arrays
    beamlike_vector = kinematics.missing_mass_technique(target_material)
    ex = beamlike_vector.mass - residual.mass
    # IMPROVE THIS ----------------------------------------------------------------------------!!!! 
    targetlike_vector = kinematics.construct_targetlike_vector()
    parent_vector = kinematics.construct_parent_vector(target_material)
    target_vector = kinematics.construct_target_vector()

    cos_polar_cm, ex_beamlike = angular_distribution(parent_vector, target_vector, targetlike_vector, residual)
    #------------------------------------------------------------------------------

    # Save information 
    results["run"] = np.append(results["run"],np.full(len(ex),run))
    results["event"] = np.append(results["event"], ex)
    results["ex"] = np.append(results["ex"],ex)
    results["polar_cm"] = np.append(results["polar_cm"],np.arccos(cos_polar_cm))

    kinematics.extract_kinematics_values(results)


    # fill histograms 
    grammer.fill_hist1d("ex",ex)

print("Files without data", empty_files)


In [ ]:
# Make angle cuts 
make_cuts = False
show_kinematic_line = True 
residual_excitations = np.array([0,0.4291,4.57,6.73])#,7.21,9.27,9.9,11.01,17]) #7Be
# mask = (results["polar_cm"]> np.deg2rad(10) ) & (results["polar_cm"]< np.deg2rad(40) ) # cm cuts 
mask = (results["polar"]>np.deg2rad(30)) & (results["polar"]<np.deg2rad(40)) # lab cuts



if not make_cuts: 
    mask = np.full(len(results["ex"]), True)


plt.close("all")
fig, axs = plt.subplots(1,3,figsize=(16,6))
fig.suptitle(f"{nuclear_map.get_data(4,7).get_latex_rep()}({nuclear_map.get_data(1,2).get_latex_rep()},{nuclear_map.get_data(1,2).get_latex_rep()}){nuclear_map.get_data(4,7).get_latex_rep()}")

# Figure 1: Kinematics
ax = axs[0]
grammer.add_hist2d("kinematics", (200,200), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics", np.rad2deg(results["polar"][mask]), results["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")

if show_kinematic_line:
    cmap = mpl.colormaps["rainbow"]
    colors = cmap(np.linspace(0,1,len(residual_excitations)))

    for idx, ex in enumerate(residual_excitations):
        ax.plot(np.rad2deg(eject_kinematics[idx, :len(cm_angle_range), 0]), eject_kinematics[idx, :len(cm_angle_range), 1], label=fr"$E_x$={ex:02} MeV",color=colors[idx])
        ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],color=colors[idx],alpha=0.2)#, label=f"Ex={ex:02} Band", alpha=0.2)

ax.grid()
ax.legend()
# Figure 2: Ex vs Polar (cm) 
ax = axs[1]
grammer.add_hist2d("ex_polarcm",(200,200), ((0,180),(-5,20)))
grammer.fill_hist2d("ex_polarcm",np.rad2deg(results["polar_cm"][mask]), results["ex"][mask])
hist = grammer.get_hist2d("ex_polarcm")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
ax.set_ylim([-5,20])
ax.set_xlim([0,180])
# fig.colorbar(mesh)
ax.set_xlabel(r"$\theta_\text{cm}$ (deg)")
ax.set_ylabel(r"$E_x$ (MeV)")
ax.grid()

if show_kinematic_line: 
    for idx, ex in enumerate(residual_excitations):
        if ex>20: continue
        ax.axhline(ex,label=fr"$E_x$={ex:02} MeV",color=colors[idx] )


# Figure 3: Ex 
ax = axs[2]
min_bin, max_bin = -5 , 14
bin_size = 38
granularity = (max_bin - min_bin) / bin_size
# missing mass 
grammer.add_hist1d("ex", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex", results["ex"][mask])
hist  = grammer.get_hist1d("ex")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black',label='missing')
# ax.scatter(hist_x, hist.counts,s=10,color='black', label='missing')

ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"$E_x$ (MeV)")

if show_kinematic_line: 
    # colors = cmap(np.linspace(0,1,len(residual_excitations)))
    for idx, ex in enumerate(residual_excitations):
        ax.axvline(ex,label=fr"$E_x$={ex:02} MeV",color=colors[idx] )

ax.legend()
# ax.set_ylim([0,25000])
ax.set_ylim([0,2000])
# ax.set_xlim([-5,4])
fig.tight_layout()
fig.savefig("delete.png")